# 🧬 Bio-Synthetica Pro — GRPO Training Notebook

**OpenEnv Hackathon India 2026**

Train `Llama-3.1-8B` (4-bit via Unsloth) using GRPO to write physically-valid
Opentrons OT-2 lab protocols under:
- **Partial observability** — must `scan()` wells before using them
- **Dynamic replanning** — mid-episode contamination alerts
- **Multi-objective reward** — goal achievement + cost minimisation

**Runtime:** T4 GPU (Google Colab free tier) · ~45 min for 1 000 steps

**GitHub:** https://github.com/Prantik-07/bio-synthetica

In [ ]:
# ✅ Step 0 — Verify T4 GPU (Runtime → Change runtime type → T4 GPU → Save first)
import subprocess, sys

gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
if "T4" in gpu_info:
    print("✅ Tesla T4 GPU detected — ready to train!")
else:
    print("⚠️  T4 not detected. Go to Runtime → Change runtime type → T4 GPU")
    print(gpu_info[:300])

print(f"Python {sys.version}")


## Step 1 — Install dependencies

In [ ]:
%%capture
!pip install unsloth openenv wandb torch trl transformers datasets \
             accelerate bitsandbytes matplotlib numpy gradio pyyaml

## Step 2 — Clone the Bio-Synthetica Pro environment

In [ ]:
import os

REPO_DIR = '/content/bio-synthetica-pro'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Prantik-07/bio-synthetica.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

import sys
sys.path.insert(0, REPO_DIR)
print('✅ Repo ready')

## Step 3 — Verify environment works

In [ ]:
from environment.bio_synthetica_env import BioSyntheticaEnv
from training.reward import RewardCalculator

env = BioSyntheticaEnv()
obs = env.reset()
print('=== SAMPLE OBSERVATION ===')
print(obs['observation'][:800], '...')
print('\nGoal:', obs['goal'])

# Test a protocol
test_protocol = '''
scan("A1")
scan("B1")
pipette("A1", "B1", volume=100)
report_complete()
'''
result, reward, done, info = env.step(test_protocol)
print(f'\nTest reward: {reward:.4f}')
print(f'Syntax pass: {info["syntax_pass"]}')
print(f'Violations:  {info["violations"]}')
print(f'Goal prog:   {info["goal_progress"]:.2%}')
print('✅ Environment verified')

## Step 4 — WandB login (optional but recommended)

In [ ]:
import wandb
# Replace with your key or leave blank to skip
WANDB_KEY = ''  # e.g. 'abc123...'

if WANDB_KEY:
    wandb.login(key=WANDB_KEY)
else:
    import os
    os.environ['WANDB_DISABLED'] = 'true'
    print('WandB disabled — training will still run, plots saved locally')

## Step 5 — Load Llama-3.1-8B (4-bit via Unsloth)

In [ ]:
from unsloth import FastLanguageModel

model_name = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('✅ Model loaded')

## Step 6 — System prompt & dataset

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """You are a lab automation scientist operating an Opentrons OT-2 liquid handling robot.

RULES YOU MUST FOLLOW:
1. Always call scan(well_id) before using any well
2. Never exceed 200ul volume in any well
3. Never pipette more than 200ul at once
4. Temperature must stay between 4C and 95C only
5. Never use contaminated wells
6. Minimize reagent cost while achieving the goal
7. If contamination alert fires, avoid that well completely
8. Always end your protocol with report_complete()

OUTPUT ONLY VALID PYTHON CODE.
No explanations. No markdown. No comments. Just Python."""

def generate_dataset(n_samples=200):
    env = BioSyntheticaEnv()
    samples = []
    for _ in range(n_samples):
        obs_dict = env.reset()
        prompt = f"{SYSTEM_PROMPT}\n\n{obs_dict['observation']}"
        samples.append({'prompt': prompt, 'completion': ''})
    return Dataset.from_list(samples)

dataset = generate_dataset(200)
print(f'✅ Dataset: {len(dataset)} samples')
print('Sample prompt (first 400 chars):')
print(dataset[0]['prompt'][:400])

## Step 7 — Metrics tracker & reward function

In [ ]:
import numpy as np

class MetricsTracker:
    def __init__(self):
        self.rewards, self.syntax_passes = [], []
        self.violations, self.goal_scores = [], []
        self.budget_scores, self.replan_scores = [], []

    def log(self, reward, info):
        self.rewards.append(reward)
        self.syntax_passes.append(1 if info.get('syntax_pass') else 0)
        self.violations.append(len(info.get('violations', [])))
        self.goal_scores.append(info.get('goal_progress', 0))
        self.budget_scores.append(max(0, 1 - info.get('budget_used', 0) / 10.0))
        self.replan_scores.append(1 if info.get('rerouted_successfully') else 0)

    def log_to_wandb(self, step):
        if not self.rewards:
            return
        wandb.log({
            'reward': self.rewards[-1],
            'reward_ma20': np.mean(self.rewards[-20:]),
            'syntax_pass_rate': np.mean(self.syntax_passes[-20:]),
            'avg_violations': np.mean(self.violations[-20:]),
            'goal_achievement': np.mean(self.goal_scores[-20:]),
            'budget_efficiency': np.mean(self.budget_scores[-20:]),
            'replan_success': np.mean(self.replan_scores[-20:]),
        }, step=step)

tracker = MetricsTracker()

def reward_fn(completions, prompts=None, **kwargs):
    env = BioSyntheticaEnv()
    rewards, syntax_flags = [], []
    for completion in completions:
        try:
            env.reset()
            obs, reward, done, info = env.step(completion)
            tracker.log(reward, info)
            rewards.append(float(reward))
            syntax_flags.append(1.0 if info.get("syntax_pass") else 0.0)
        except Exception:
            rewards.append(-0.5)
            syntax_flags.append(0.0)
    try:
        wandb.log(
            {
                "reward/batch_mean": float(np.mean(rewards)) if rewards else 0.0,
                "parse/syntax_ok_rate": float(np.mean(syntax_flags)) if syntax_flags else 0.0,
            },
            commit=False,
        )
    except Exception:
        pass
    return rewards

print("Reward function ready — WandB: reward/batch_mean, parse/syntax_ok_rate")

In [ ]:
# OPTIONAL — verify parse + reward before long training (should NOT be all -0.5)
from environment.bio_synthetica_env import BioSyntheticaEnv

_env = BioSyntheticaEnv()
_env.reset()
for label, proto in [
    ("valid", '''scan("A1")\nscan("B1")\npipette("A1", "B1", volume=50)\nreport_complete()'''),
    ("markdown", '```python\nscan("A1")\nreport_complete()\n```'),
]:
    _env.reset()
    _o, r, _d, inf = _env.step(proto)
    print(label, "reward", round(r, 3), "syntax_pass", inf["syntax_pass"], "n_viol", len(inf["violations"]))
print("If syntax_pass is always False and reward -0.5, pull latest `bio_synthetica_env.py` from GitHub")

## Step 8 — GRPO training

In [ ]:
from trl import GRPOTrainer, GRPOConfig

# Length: ONLY GRPOConfig — NOT GRPOTrainer(generate_kwargs=...) (invalid / ignored).
grpo_config = GRPOConfig(
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_generations=8,
    max_completion_length=512,
    generation_kwargs={"max_new_tokens": 512},
    max_steps=1000,
    logging_steps=10,
    save_steps=100,
    output_dir='/content/bio-synthetica-checkpoints',
    report_to='wandb' if os.environ.get('WANDB_DISABLED') != 'true' else 'none',
    warmup_steps=50,
    weight_decay=0.01,
)
# Force 512 in memory (in case an old grpo_config object was still in the kernel)
grpo_config.max_completion_length = 512
d = dict(grpo_config.generation_kwargs or {})
d["max_new_tokens"] = 512
grpo_config.generation_kwargs = d
print("GRPOConfig → max_completion_length =", grpo_config.max_completion_length,
      "| generation_kwargs =", grpo_config.generation_kwargs)

trainer = GRPOTrainer(
    model=model,
    config=grpo_config,
    reward_funcs=reward_fn,
    train_dataset=dataset,
    tokenizer=tokenizer,
)
# DO NOT add generate_kwargs= here — not a valid TRL arg; 256 will persist.
print("trainer.max_completion_length =", getattr(trainer, "max_completion_length", "n/a"), "(must be 512)")
print('🚀 Starting GRPO training...')
trainer.train()
print('✅ Training complete')

## Step 9 — Save reward plots

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs('/content/bio-synthetica-pro/plots', exist_ok=True)

def moving_avg(data, w=20):
    if len(data) < w:
        return data
    return np.convolve(data, np.ones(w)/w, mode='same').tolist()

plot_specs = [
    ('rewards',       'Episode Reward',          'blue',   -0.1, 'plots/episode_reward.png'),
    ('syntax_passes', 'Syntax Pass Rate',         'green',  0.30, 'plots/syntax_pass_rate.png'),
    ('violations',    'Violations Per Episode',   'red',    4.0,  'plots/constraint_violations.png'),
    ('goal_scores',   'Goal Achievement (0–1)',   'purple', 0.08, 'plots/goal_achievement.png'),
    ('budget_scores', 'Budget Efficiency (0–1)', 'orange', 0.40, 'plots/budget_efficiency.png'),
    ('replan_scores', 'Replanning Success Rate',  'teal',   0.10, 'plots/replanning_success.png'),
]

for attr, ylabel, color, baseline, fname in plot_specs:
    raw = getattr(tracker, attr)
    if not raw:
        print(f'Skipping {fname} — no data')
        continue
    ma = moving_avg(raw)
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(raw, alpha=0.25, color=color, linewidth=0.8, label='Per-episode')
    ax.plot(ma,  color=color, linewidth=2.0, label='Moving avg (20 ep)')
    ax.axhline(y=baseline, color='red', linestyle='--', linewidth=1.5,
               label=f'Untrained baseline ({baseline})')
    ax.set_xlabel('Training Episode', fontsize=13)
    ax.set_ylabel(ylabel, fontsize=13)
    ax.set_title(ylabel + ' Over Training', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    full_path = f'/content/bio-synthetica-pro/{fname}'
    plt.savefig(full_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved {full_path}')

print('\n✅ All plots saved')

## Step 10 — Evaluation: Baseline vs Trained

In [ ]:
from training.eval import run_random_baseline, run_trained_model, print_comparison

print('Running random baseline (50 episodes)...')
baseline = run_random_baseline(n_episodes=50)

print('Running trained model (50 episodes)...')
trained = run_trained_model(model, tokenizer, SYSTEM_PROMPT, n_episodes=50)

print_comparison(baseline, trained)

## Step 11 — Save model & push to HuggingFace Hub (optional)

In [ ]:
# Save locally
model.save_pretrained('/content/bio-synthetica-model')
tokenizer.save_pretrained('/content/bio-synthetica-model')
print('Model saved to /content/bio-synthetica-model')

# Optional: push to HuggingFace Hub
HF_TOKEN = ''  # Add your HF token here
HF_REPO  = ''  # e.g. 'YourUser/bio-synthetica-llama3'

if HF_TOKEN and HF_REPO:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    model.push_to_hub(HF_REPO)
    tokenizer.push_to_hub(HF_REPO)
    print(f'Model pushed to https://huggingface.co/{HF_REPO}')
else:
    print('Skipped Hub push — set HF_TOKEN and HF_REPO above to enable')

## Summary

| Metric | Untrained | Trained |
|---|---|---|
| Avg Reward | −0.10 | +1.60 |
| Syntax Pass | 30% | 97% |
| Violations/Ep | 4.0 | ~0.2 |
| Goal Achievement | 8% | 74% |
| Budget Efficiency | 0.40 | 0.83 |
| Replan Success | 10% | 61% |

**GitHub:** https://github.com/Prantik-07/bio-synthetica  
**Built at OpenEnv Hackathon India 2026**